# Resistencia antimicrobiana (AMR) — Unión, EDA y limpieza (2012–2024)

En este notebook unificamos los CSV de **resistencia antimicrobiana** por **bacteria**, **grupo antibiótico**, **país** y **año**.

**Objetivos:**
1. Cargar y concatenar únicamente los CSV de **resistencia** desde `../data/raw`
2. Mantener columnas relevantes y renombrarlas
3. Ejecutar EDA estructurado para validar calidad
4. Aplicar limpieza mínima (valores faltantes y tipos)
5. Exportar el dataset final a `../data/processed`

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# ==============================
# 1) Ruta de entrada (raw)
# ==============================
RAW_DIR = Path("../data/raw")

# ==============================
# 2) Detectar SOLO CSV de resistencia
#    (en tu carpeta, los de resistencia tienen '-' en el nombre)
# ==============================
files = [
    f for f in os.listdir(RAW_DIR)
    if f.lower().endswith(".csv") and "-" in f
]

print(f"📂 CSV de resistencia detectados en {RAW_DIR}: {len(files)}")
display(sorted(files))

# ==============================
# 3) Cargar todos los CSV en una lista
# ==============================
dfs = []
for file in files:
    df_temp = pd.read_csv(RAW_DIR / file)
    dfs.append(df_temp)

# ==============================
# 4) Concatenar por filas
# ==============================
df_raw_amr = pd.concat(dfs, ignore_index=True)

print("✅ Dataset concatenado (raw AMR)")
print("Shape:", df_raw_amr.shape)
display(df_raw_amr.head())

📂 CSV de resistencia detectados en ..\data\raw: 11


['acinetobacter-aminoglyc.csv',
 'acinetobacter-carbap.csv',
 'acinetobacter-fluoro.csv',
 'coli-aminoglyc.csv',
 'coli-carbapenems.csv',
 'coli-cephalosporins.csv',
 'coli-fluoro.csv',
 'pneumoniae-aminoglyc.csv',
 'pneumoniae-carba.csv',
 'pneumoniae-cepha.csv',
 'pneumoniae-flouro.csv']

✅ Dataset concatenado (raw AMR)
Shape: (6161, 9)


,HealthTopic,Population,Indicator,Unit,Time,RegionCode,RegionName,NumValue,TxtValue
0,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,BG,Bulgaria,58.461538460,NaN
1,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,CY,Cyprus,52.173913040,NaN
2,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DE,Germany,5.882352940,NaN
3,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,DK,Denmark,10.389610380,NaN
4,Antimicrobial resistance,Acinetobacter spp.|Aminoglycosides,"R - resistant isolates, percentage",%,2012,EL,Greece,78.119935170,NaN


## 2) Selección y estandarización de columnas relevantes

El dataset original contiene 9 columnas.  
Para el análisis nos quedamos únicamente con las necesarias:

- Population → información de bacteria + grupo antibiótico
- Time → año
- RegionCode → código país
- RegionName → país
- NumValue → porcentaje de resistencia

Renombramos las columnas para facilitar su uso posterior.

In [4]:
# ==========================================================
# 1) Seleccionar columnas relevantes
# ==========================================================
df_amr = df_raw_amr[[
    "Population",
    "Time",
    "RegionCode",
    "RegionName",
    "NumValue"
]].copy()

print("Columnas seleccionadas:")
display(df_amr.head())


# ==========================================================
# 2) Renombrar columnas
# ==========================================================
df_amr = df_amr.rename(columns={
    "Population": "bacteria_antibiotic",
    "Time": "year",
    "RegionCode": "country_code",
    "RegionName": "country",
    "NumValue": "resistance_percentage"
})

print("Columnas renombradas:")
display(df_amr.head())


# ==========================================================
# 3) Reordenar columnas (orden lógico)
# ==========================================================
df_amr = df_amr[
    ["year", "country_code", "country", "bacteria_antibiotic", "resistance_percentage"]
]

print("Dataset tras selección y renombrado:")
print("Shape:", df_amr.shape)
display(df_amr.head())

Columnas seleccionadas:


,Population,Time,RegionCode,RegionName,NumValue
0,Acinetobacter spp.|Aminoglycosides,2012,BG,Bulgaria,58.461538460
1,Acinetobacter spp.|Aminoglycosides,2012,CY,Cyprus,52.173913040
2,Acinetobacter spp.|Aminoglycosides,2012,DE,Germany,5.882352940
3,Acinetobacter spp.|Aminoglycosides,2012,DK,Denmark,10.389610380
4,Acinetobacter spp.|Aminoglycosides,2012,EL,Greece,78.119935170


Columnas renombradas:


,bacteria_antibiotic,year,country_code,country,resistance_percentage
0,Acinetobacter spp.|Aminoglycosides,2012,BG,Bulgaria,58.461538460
1,Acinetobacter spp.|Aminoglycosides,2012,CY,Cyprus,52.173913040
2,Acinetobacter spp.|Aminoglycosides,2012,DE,Germany,5.882352940
3,Acinetobacter spp.|Aminoglycosides,2012,DK,Denmark,10.389610380
4,Acinetobacter spp.|Aminoglycosides,2012,EL,Greece,78.119935170


Dataset tras selección y renombrado:
Shape: (6161, 5)


,year,country_code,country,bacteria_antibiotic,resistance_percentage
0,2012,BG,Bulgaria,Acinetobacter spp.|Aminoglycosides,58.461538460
1,2012,CY,Cyprus,Acinetobacter spp.|Aminoglycosides,52.173913040
2,2012,DE,Germany,Acinetobacter spp.|Aminoglycosides,5.882352940
3,2012,DK,Denmark,Acinetobacter spp.|Aminoglycosides,10.389610380
4,2012,EL,Greece,Acinetobacter spp.|Aminoglycosides,78.119935170


## 3) Separación de bacteria y antibiótico

La columna `bacteria_antibiotic` contiene información combinada en el formato:

Bacteria | Grupo antibiótico

Para facilitar el análisis y permitir filtros independientes en Tableau,
separamos esta columna en:

- `bacteria`
- `antibiotic_group`

Posteriormente eliminamos la columna original.

In [5]:
# ==========================================================
# 1) Separar bacteria y antibiótico usando el separador "|"
# ==========================================================
df_amr[["bacteria", "antibiotic_group"]] = (
    df_amr["bacteria_antibiotic"]
    .str.split("|", expand=True)
)

# ==========================================================
# 2) Limpieza básica de texto (quitar espacios)
# ==========================================================
df_amr["bacteria"] = df_amr["bacteria"].str.strip()
df_amr["antibiotic_group"] = df_amr["antibiotic_group"].str.strip()

# ==========================================================
# 3) Eliminar la columna original combinada
# ==========================================================
df_amr = df_amr.drop(columns=["bacteria_antibiotic"])

# ==========================================================
# 4) Reordenar columnas finales
# ==========================================================
df_amr = df_amr[
    ["year", "country_code", "country", "bacteria", "antibiotic_group", "resistance_percentage"]
]

print("✅ Columnas tras separación:")
display(df_amr.head())

print("\nColumnas actuales:")
print(df_amr.columns.tolist())

print("\nShape actual:", df_amr.shape)

✅ Columnas tras separación:


,year,country_code,country,bacteria,antibiotic_group,resistance_percentage
0,2012,BG,Bulgaria,Acinetobacter spp.,Aminoglycosides,58.461538460
1,2012,CY,Cyprus,Acinetobacter spp.,Aminoglycosides,52.173913040
2,2012,DE,Germany,Acinetobacter spp.,Aminoglycosides,5.882352940
3,2012,DK,Denmark,Acinetobacter spp.,Aminoglycosides,10.389610380
4,2012,EL,Greece,Acinetobacter spp.,Aminoglycosides,78.119935170



Columnas actuales:
['year', 'country_code', 'country', 'bacteria', 'antibiotic_group', 'resistance_percentage']

Shape actual: (6161, 6)


## 4) Limpieza y validaciones del dataset de resistencia

En esta fase verificamos que el dataset cumple criterios mínimos de calidad:

- Conversión de tipos
- Control de valores nulos
- Verificación del rango de años
- Detección de valores fuera de rango (0–100%)
- Comprobación de duplicados estructurales

El objetivo es garantizar que el dataset esté listo para análisis conjunto con el consumo.

In [6]:
def clean_amr_dataset(df):
    """
    Limpieza y validación del dataset de resistencia antimicrobiana (AMR).

    Acciones realizadas:
    - Conversión de tipos (year y resistance_percentage)
    - Limpieza básica de texto
    - Validaciones estructurales
    - Comprobación de duplicados

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame de resistencia ya transformado.

    Returns
    -------
    pandas.DataFrame
        DataFrame validado y limpio.
    """

    df_clean = df.copy()

    # ==========================================================
    # 1) Conversión de tipos
    # ==========================================================
    df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce").astype("Int64")
    df_clean["resistance_percentage"] = pd.to_numeric(
        df_clean["resistance_percentage"], errors="coerce"
    )

    # ==========================================================
    # 2) Limpieza básica de texto
    # ==========================================================
    df_clean["country"] = df_clean["country"].astype(str).str.strip()
    df_clean["bacteria"] = df_clean["bacteria"].astype(str).str.strip()
    df_clean["antibiotic_group"] = df_clean["antibiotic_group"].astype(str).str.strip()

    # ==========================================================
    # 3) VALIDACIONES
    # ==========================================================
    print("=" * 90)
    print("🧼 LIMPIEZA — VALIDACIONES AMR")
    print("=" * 90)

    # 3.1 Nulos
    print("\n🚨 Nulos por columna:")
    display(df_clean.isnull().sum())

    # 3.2 Rango de años
    years_sorted = sorted(df_clean["year"].dropna().unique())
    print("\n📅 Años detectados:")
    display(years_sorted)

    if len(years_sorted) > 0:
        print(f"Rango de años: {years_sorted[0]}–{years_sorted[-1]}")

    # 3.3 Valores fuera de rango lógico (0–100%)
    n_neg = (df_clean["resistance_percentage"] < 0).sum()
    n_over_100 = (df_clean["resistance_percentage"] > 100).sum()

    print(f"\n📉 Valores negativos en resistencia: {n_neg}")
    print(f"📈 Valores > 100%: {n_over_100}")

    # 3.4 Duplicados estructurales
    key_cols = ["year", "country", "bacteria", "antibiotic_group"]
    dup_count = df_clean.duplicated(subset=key_cols).sum()
    print(f"\n🔁 Duplicados por clave {key_cols}: {dup_count}")

    print("\nShape final:", df_clean.shape)

    return df_clean


# Ejecutar limpieza
df_amr_clean = clean_amr_dataset(df_amr)

🧼 LIMPIEZA — VALIDACIONES AMR

🚨 Nulos por columna:


year                       0
country_code               0
country                    0
bacteria                   0
antibiotic_group           0
resistance_percentage    262
dtype: int64


📅 Años detectados:


[np.int64(2000),
 np.int64(2001),
 np.int64(2002),
 np.int64(2003),
 np.int64(2004),
 np.int64(2005),
 np.int64(2006),
 np.int64(2007),
 np.int64(2008),
 np.int64(2009),
 np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024)]

Rango de años: 2000–2024

📉 Valores negativos en resistencia: 0
📈 Valores > 100%: 0

🔁 Duplicados por clave ['year', 'country', 'bacteria', 'antibiotic_group']: 0

Shape final: (6161, 6)


## 5) Alineación temporal y gestión de valores faltantes

El dataset de resistencia cubre el periodo 2000–2024.
Sin embargo, el dataset de consumo cubre únicamente 2012–2024.

Para garantizar coherencia en el análisis conjunto:

- Filtramos resistencia al rango 2012–2024
- Mantenemos los valores NaN en resistencia_percentage
  (representan datos no reportados y no deben imputarse)

In [7]:
# ==========================================================
# 1) Filtrar rango temporal (2012–2024)
# ==========================================================
df_amr_filtered = df_amr_clean[
    (df_amr_clean["year"] >= 2012) &
    (df_amr_clean["year"] <= 2024)
].copy()

print("📅 Dataset filtrado por rango temporal 2012–2024")
print("Shape tras filtro:", df_amr_filtered.shape)

years_sorted = sorted(df_amr_filtered["year"].dropna().unique())
print("Años finales:", years_sorted)


# ==========================================================
# 2) Revisar nulos tras filtrado
# ==========================================================
print("\n🚨 Nulos tras filtrado:")
display(df_amr_filtered.isnull().sum())

📅 Dataset filtrado por rango temporal 2012–2024
Shape tras filtro: (4186, 6)
Años finales: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

🚨 Nulos tras filtrado:


year                       0
country_code               0
country                    0
bacteria                   0
antibiotic_group           0
resistance_percentage    185
dtype: int64

## 6) Creación de columna ATC común para integración con consumo

Para poder analizar conjuntamente consumo y resistencia,
creamos una nueva columna `atc_group` que mapea los grupos de resistencia
a los grupos ATC utilizados en el dataset de consumo.

Reglas de mapeo:

- Aminoglycosides → J01G
- Carbapenems → J01D
- Third-generation cephalosporins → J01D
- Fluoroquinolones → J01M

Esta columna permitirá relacionar ambos datasets directamente.

In [9]:
# ==========================================================
# Crear diccionario de mapeo (versión corregida)
# ==========================================================
atc_mapping = {
    "Aminoglycosides": "J01G",
    "Carbapenems": "J01D",
    "Third-generation cephalosporins": "J01D",
    "Fluoroquinolones": "J01M"
}

# ==========================================================
# Aplicar mapeo
# ==========================================================
df_amr_filtered["atc_group"] = df_amr_filtered["antibiotic_group"].map(atc_mapping)

# ==========================================================
# Validación final
# ==========================================================
print("Recuento por antibiotic_group original:")
display(df_amr_filtered["antibiotic_group"].value_counts())

print("\nRecuento por atc_group nuevo:")
display(df_amr_filtered["atc_group"].value_counts())

print("\nValores sin mapear (si existen):")
display(df_amr_filtered[df_amr_filtered["atc_group"].isnull()]["antibiotic_group"].unique())

print("\nShape final:", df_amr_filtered.shape)
display(df_amr_filtered.head())

Recuento por antibiotic_group original:


antibiotic_group
Aminoglycosides                    1140
Fluoroquinolones                   1140
Carbapenems                        1133
Third-generation cephalosporins     773
Name: count, dtype: int64


Recuento por atc_group nuevo:


atc_group
J01D    1906
J01G    1140
J01M    1140
Name: count, dtype: int64


Valores sin mapear (si existen):


array([], dtype=object)


Shape final: (4186, 7)


,year,country_code,country,bacteria,antibiotic_group,resistance_percentage,atc_group
0,2012,BG,Bulgaria,Acinetobacter spp.,Aminoglycosides,58.461538,J01G
1,2012,CY,Cyprus,Acinetobacter spp.,Aminoglycosides,52.173913,J01G
2,2012,DE,Germany,Acinetobacter spp.,Aminoglycosides,5.882353,J01G
3,2012,DK,Denmark,Acinetobacter spp.,Aminoglycosides,10.389610,J01G
4,2012,EL,Greece,Acinetobacter spp.,Aminoglycosides,78.119935,J01G


## 7) Exportación final del dataset de resistencia (versión integrada)

Se exporta la versión limpia y alineada (2012–2024),
incluyendo la columna `atc_group` que permite integrar
directamente con el dataset de consumo.

Este archivo constituye la versión oficial procesada del dataset AMR.

In [10]:
# ==============================
# Exportación definitiva
# ==============================
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "antimicrobial_resistance_2012_2024_processed.csv"
df_amr_filtered.to_csv(output_path, index=False)

print(f"✅ Archivo exportado en: {output_path}")
print("Filas exportadas:", df_amr_filtered.shape[0])
print("Columnas exportadas:", df_amr_filtered.shape[1])

✅ Archivo exportado en: ..\data\processed\antimicrobial_resistance_2012_2024_processed.csv
Filas exportadas: 4186
Columnas exportadas: 7
